# One-click faculty review
The first cell automatically selects the repository root.

In [1]:
# Make imports work whether VS Code starts in the repository or notebooks folder.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError(f"Cannot find the project root from {Path.cwd()}")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

Project root: c:\Users\jahav\OneDrive\janudrive\project\EV-Charging-RL-Optimizer


In [2]:
from src.utils.config import load_config
from src.data.prepare_data import prepare_data
config=load_config('review'); stations,report=prepare_data(config=config); display(report)

{'input_rows': 855,
 'output_rows': 500,
 'invalid_coordinate_rows_removed': 0,
 'duplicate_rows_removed': 355,
 'missing_power_filled': 1,
 'source_url': 'https://www.kaggle.com/datasets/pranjal9091/ev-charging-stations-in-india-simplified-2025'}

In [3]:
from src.network.station_graph import build_station_graph
graph=build_station_graph(stations.head(15),config['network']['edge_radius_km']); print(graph)

Graph with 15 nodes and 36 edges


In [4]:
from src.environment.ev_charging_env import EVChargingEnv
from src.baselines.policies import nearest_action,weighted_greedy_action
env=EVChargingEnv(stations,config,debug=True); observation,info=env.reset(seed=42); display(env.candidates)

,station_id,station_name,city,state,latitude,longitude,operator,usage_type,connector_type,power_kw,power_kw_source,number_of_chargers,number_of_chargers_source,base_price_inr_kwh,base_price_source,distance_km
0,IND-0053,Lulu International Mall EVCS - GO EC,Thiruvananthapuram,Kerala,8.515263,76.898158,GO EC (IN),Public - Membership Required,CCS Type 2,60.0,real_dataset,2,simulated_default,18.0,simulated_default,0.542320
1,IND-0370,ANERT - EESL Sangamukham - EESL,Thiruvananthapuram,Kerala,8.481015,76.912643,EESL (IN),Public - Membership Required,CCS Type 2,60.0,real_dataset,2,simulated_default,18.0,simulated_default,3.658099
2,IND-0445,Trivandrum Airport T1,Thiruvananthapuram,Kerala,8.476058,76.918310,Adani Gas-EV (TR),Public - Membership Required,CCS Type 2,30.0,real_dataset,2,simulated_default,18.0,simulated_default,4.399486
3,IND-0443,Hilton Garden Inn Trivandrum,Thiruvananthapuram,Kerala,8.499635,76.950299,Statiq (IN),Public - Membership Required,CCS Type 2,60.0,real_dataset,2,simulated_default,18.0,simulated_default,5.494094
4,IND-0442,JKV Power Hub,Thiruvananthapuram,Kerala,8.503668,76.953916,ChargeMod (IN),Public - Membership Required,CCS Type 2,30.0,real_dataset,2,simulated_default,18.0,simulated_default,5.789759
5,IND-0052,Appolo Dimora Hotel EVCS - GO EC,Thiruvananthapuram,Kerala,8.488661,76.950752,GO EC (IN),Public - Membership Required,CCS Type 2,30.0,real_dataset,2,simulated_default,18.0,simulated_default,5.967373
6,IND-0444,CBC Motors Charging Station,Thiruvananthapuram,Kerala,8.465872,76.939098,Statiq (IN),Public - Membership Required,CCS Type 2,60.0,real_dataset,2,simulated_default,18.0,simulated_default,6.570884
7,IND-0390,Marvel Paints - ChargeZone,Thiruvananthapuram,Kerala,8.535706,76.965262,Chargezone (India),Public - Membership Required,CCS Type 2,30.0,real_dataset,2,simulated_default,18.0,simulated_default,7.435009
8,IND-0440,Powerhub EVCS Attukal,Thiruvananthapuram,Kerala,8.466545,76.960808,ChargeMod (IN),Public - Membership Required,CCS Type 2,30.0,real_dataset,2,simulated_default,18.0,simulated_default,8.222345
9,IND-0441,Sleeba & Sons Station,Thiruvananthapuram,Kerala,8.544983,76.972298,Statiq (IN),Public - Membership Required,CCS Type 2,60.0,real_dataset,2,simulated_default,18.0,simulated_default,8.550384


In [5]:
recommendation=weighted_greedy_action(env); alternatives=[i for i,v in enumerate(env.candidate_mask) if v and i!=recommendation]; print('Recommended index:',recommendation,'alternatives:',alternatives[:3])

Recommended index: 0 alternatives: [1, 2, 3]


In [6]:
before=env.queue_manager.snapshot(); actual=alternatives[0] if alternatives else recommendation; env.set_actual_selection(actual); _,reward,_,_,result=env.step(recommendation); after=env.queue_manager.snapshot(); print('Recommended:',result['recommended_station_id']); print('Actual:',result['actual_selected_station_id']); print('Accepted:',result['recommendation_accepted']); print('Reward:',reward)

STEP 1
EV SOC: 45.4%  Required Energy: 31.5 kWh  Safe Range: 151.5 km
ENVIRONMENT Weather severity: 0.35  Traffic: 0.32  Hour: 2
ACTION Recommended index: 0  Actual index: 1
RESULT Travel: 10.0  Wait: 0.0  Charge: 48.6  Cost: 799.0
REWARD 90.19 {'waiting': 100.0, 'travel': 91.67450993254272, 'cost': 67.0978339405113, 'utilization': 100.0, 'balance': 87.52780871075353, 'demand': 80.41877262828203, 'compatibility': 100.0}
Recommended: IND-0053
Actual: IND-0370
Accepted: False
Reward: 90.1942352114888


In [7]:
changed=[station for station in after if before[station]!=after[station]]; print('Changed stations:',changed); assert result['actual_selected_station_id'] in changed

Changed stations: ['IND-0370']
